This notebook takes the output of the ortho creation step & precomputes wald

In [ ]:
import scMPRAforge as scm
from dask_jobqueue import SLURMCluster
from dask.distributed import Client, LocalCluster

Set up the cluster

In [ ]:
local=False
if local:
    cluster=LocalCluster(memory_limit='48G')
    client = Client(cluster)
else:
    cluster=SLURMCluster(
        cores=8,#cores per slurm job
        memory="80G",#memory per slurm job
        processes=1,#dask workers per slurm job
        job_extra_directives=["-p ycga", 
            f"--job-name=simclust_worker",
            f"--time=1:00:00",
            f"--output=slave_%j.out"]
    )
    cluster.scale(jobs=3)
    client = Client(cluster,
            timeout=f"{5*60}s",   # Client <-> scheduler timeout 
            heartbeat_interval="20s"  # Worker heartbeat interval
        )

Let's just do a super simple bounded concurrency approach

In [ ]:
from pathlib import Path

In [ ]:
# in the real version, de_novo_sim will take a pair, path/name on init and never save it.
# relative paths to individual components can be used, saved, assumed. 
DATA_ROOT=Path("/gpfs/gibbs/pi/reilly/tabula_data")
path=DATA_ROOT/"simulated"
name="shendure_calibrated_sim_with_orthos_20251008"

In [ ]:
from dask.distributed import Semaphore, as_completed, get_client

In [ ]:
def chkdir(path):
    p = Path(path)
    if not p.is_dir():
        raise FileNotFoundError(f"Directory not found: {p}")

def chkfile(path):
    p = Path(path)
    if not p.is_file():
        raise FileNotFoundError(f"File not found: {p}")

In [ ]:
ortho_root=path/name/"orthos"
scmpradat_root=path/name/"scMPRA"
output_root=path/name/"orthos_with_precomputed_wald"
output_root.mkdir(exist_ok=True)

input_ortho_names=[path.name for path in ortho_root.iterdir()]

Semaphore(max_leases=2, name="wald-precompute")

def precompute_one_wald(input_root, scmpradat_root, name, output_root):
    sem = Semaphore(name="wald-precompute")
    with sem:
        client=get_client()
        dat=scm.scMPRA_data.from_parquet(scmpradat_root/Path(name).with_suffix(".scmpra"))
        dat.ortho_filter()
        ortho_oi=scm.ortho.load(client=client,
                                path=input_root,
                                name=name)
        ortho_oi.training_data=dat
        ortho_oi.precompute_wald(client)
        ortho_oi.save(path=output_root,name=name)

futures = [client.submit(precompute_one_wald, input_root=ortho_root,scmpradat_root=scmpradat_root,name=name_oi,output_root=output_root) for name_oi in input_ortho_names]

In [ ]:
#futures=[]

In [ ]:
#futures.append(client.submit(precompute_one_wald, input_root=ortho_root,scmpradat_root=scmpradat_root,name=input_ortho_names[0],output_root=output_root))

In [ ]:
for fut in as_completed(futures):
    print(fut.result())

In [ ]:
client.close()
cluster.close()